# Notebook 02 — Preprocessing
## STOP 4 — Train/Test Split
Split data with stratification.

In [11]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import torch
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = ROOT / 'data' / 'raw' / 'heart.csv'
PROC = ROOT / 'data' / 'processed'
ARTIFACTS = ROOT / 'data' / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True, parents=True)
PROC.mkdir(exist_ok=True, parents=True)

df = pd.read_csv(DATA_PATH)
X = df.drop(columns=['HeartDisease'])
y = df['HeartDisease']

categorical_cols = [c for c in X.columns if X[c].dtype == 'object']
numeric_cols = [c for c in X.columns if c not in categorical_cols]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('train class ratio:', y_train.value_counts(normalize=True))
print('test class ratio:', y_test.value_counts(normalize=True))

train class ratio: HeartDisease
1    0.553134
0    0.446866
Name: proportion, dtype: float64
test class ratio: HeartDisease
1    0.554348
0    0.445652
Name: proportion, dtype: float64


## STOP 5 — Feature Scaling
Fit scaler on train only, transform train/test, and save scaler.

In [12]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categorical_cols),
])

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

with open(ARTIFACTS / 'preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
with open(ARTIFACTS / 'scaler.pkl', 'wb') as f:
    pickle.dump(preprocessor.named_transformers_['num'], f)

print('saved scaler:', ARTIFACTS / 'scaler.pkl')

saved scaler: /home/ashutosh/Desktop/NO-AI-USE/Deep-learning/ANN Binary Classification/data/processed/scaler.pkl


## STOP 6 — Convert to PyTorch Tensors
Create FloatTensors, TensorDataset, and DataLoader.

In [13]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy(dtype=np.float32)).unsqueeze(1)
y_test_t = torch.tensor(y_test.to_numpy(dtype=np.float32)).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32, shuffle=False)

print(X_train_t.shape, y_train_t.shape)
print(X_test_t.shape, y_test_t.shape)

torch.Size([734, 15]) torch.Size([734, 1])
torch.Size([184, 15]) torch.Size([184, 1])
